# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/-Explaining-Search-Performance-Gaps-Using-Ranking-Signals/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Lane 1 + Lane 4 – *Explaining Search Performance Gaps Using Ranking Signals*

I chose to combine Lane 1 and Lane 4 because identifying pages with performance gaps is much more valuable when we can also explain why those gaps exist. My goal is to analyze ranking signals—such as content age, freshness, word count, content type, search intent, and other ranking-related signals available in the starter dataset content quality indicators—and connect them to pages that receive fewer clicks or perform below expectations. This approach focuses on finding actionable opportunities, helping turn search performance data into clear recommendations for improving rankings and traffic.


## 2. The question: decision, action, cost of a wrong call

My work addresses three connected questions:

1. **Where is the gap?**
   Which pages are underperforming relative to their ranking position?

2. **Why does the gap exist?**
   Which ranking signals — schema, readability, content structure —
   are missing or weak in those pages?

3. **What should we do about it?**
   For existing pages: which specific signals to fix, in what order,
   to close the gap?
   For new pages: which signals must be present from day one
   to avoid reproducing the same pattern?

**Who acts on it:**
Content writers and SEO strategists at FlyRank — both when
auditing existing pages for improvement AND when planning new ones.

**Cost of a wrong call:**
- Misidentify the cause → fix the wrong signal → gap persists,
  effort is wasted
- Miss a fixable page → underperforming page stays underperforming
  while competitors close the gap
- Wrong prevention advice → new pages launch with the same
  structural weaknesses → gap reproduces itself at scale

## 3. Quick look at the data (2-3 real numbers)



In [9]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [26]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Dataset shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


### Number 1: Pages with high position but low clicks

In [27]:

print("avg_position distribution:")
print(df["avg_position"].describe())

print("\nposition_tier value counts:")
print(df["position_tier"].value_counts())

print("\nSample of avg_position vs position_tier:")
print(df[["avg_position", "position_tier"]].drop_duplicates().sort_values("avg_position").head(10))

avg_position distribution:
count    30000.00000
mean        16.34238
std         15.21679
min          0.00000
25%          6.20000
50%         10.80000
75%         22.30000
max        245.00000
Name: avg_position, dtype: float64

position_tier value counts:
position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
Name: count, dtype: int64

Sample of avg_position vs position_tier:
       avg_position position_tier
11              0.0         top_3
4873            0.1         top_3
18992           0.2         top_3
1289            0.3         top_3
10077           0.4         top_3
1128            0.5         top_3
1697            0.6         top_3
903             0.7         top_3
5517            0.8         top_3
2984            0.9         top_3


In [34]:
print(f"df shape: {df.shape}")

print("\nposition_tier unique values:")
print(df["position_tier"].unique())


print("\nposition_tier value counts:")
print(df["position_tier"].value_counts())

print("\nRows matching 'top_3' (anywhere in string):")
print(df[df["position_tier"].str.contains("top_3", case=False, na=False)].shape)

df shape: (30000, 44)

position_tier unique values:
['striking' 'page_3_5' 'page_1' 'top_3' 'deep']

position_tier value counts:
position_tier
page_1      11814
striking     7304
page_3_5     7242
top_3        2321
deep         1319
Name: count, dtype: int64

Rows matching 'top_3' (anywhere in string):
(2321, 44)


In [36]:
top3 = df[df['position_tier'] == 'top_3']
min_impressions = 100
top3_volume = top3[top3["impressions_90d"] >= min_impressions].copy()

typical_ctr = top3_volume["clicks_90d"].sum() / top3_volume["impressions_90d"].sum()
top3_volume["expected_clicks"] = top3_volume["impressions_90d"] * typical_ctr


top3_volume["perf_ratio"] = top3_volume["clicks_90d"] / top3_volume["expected_clicks"]
top3_volume["underperforming"] = top3_volume["perf_ratio"] < 0.5


print(f"Top-3 pages (all): {len(top3):,}")
print(f"Top-3 pages with {min_impressions}+ impressions: {len(top3_volume):,}")
print(f"Underperforming (< 50% expected): {top3_volume['underperforming'].sum():,}")
print(f"Percentage: {top3_volume['underperforming'].mean()*100:.1f}%")
print(f"\nTypical CTR in top-3: {typical_ctr*100:.2f}%")


Top-3 pages (all): 2,321
Top-3 pages with 100+ impressions: 533
Underperforming (< 50% expected): 303
Percentage: 56.8%

Typical CTR in top-3: 0.49%


**Finding:** Of 533 top-3 pages with meaningful traffic (100+ impressions),
**303 (56.8%) are underperforming** — they receive less than 50% of expected clicks
based on typical top-3 CTR (0.49%).

**Interpretation:** More than half of FlyRank's top-ranked content is not capturing
clicks proportional to its position. This is the gap we aim to explain and close.

### Number 2: Content signals differ between high-click and low-click pages

In [37]:
top3_volume["perf_group"] = "medium"
top3_volume.loc[top3_volume["perf_ratio"] >= 1.5, "perf_group"] = "high"
top3_volume.loc[top3_volume["perf_ratio"] <= 0.5, "perf_group"] = "low"

print("Performance groups:")
print(top3_volume["perf_group"].value_counts())

signal_cols = [
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "scroll_rate",
]

print(f"\nSignals we'll compare: {signal_cols}")
comparison = top3_volume.groupby("perf_group")[signal_cols].median()


if "high" in comparison.index and "low" in comparison.index:
    diff = comparison.loc["high"] - comparison.loc["low"]
    pct_diff = (diff / comparison.loc["low"] * 100).round(1)

    print("\n=== Absolute difference (High - Low) ===")
    print(diff.round(2))

    print("\n=== Percentage difference ===")
    print(pct_diff)

Performance groups:
perf_group
low       303
medium    159
high       71
Name: count, dtype: int64

Signals we'll compare: ['word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'engagement_rate', 'scroll_rate']

=== Absolute difference (High - Low) ===
word_count                 19.50
char_count                153.50
content_age_days          178.00
days_since_last_update      0.00
engagement_rate             2.75
scroll_rate                 5.56
dtype: float64

=== Percentage difference ===
word_count                  0.7
char_count                  0.8
content_age_days          117.9
days_since_last_update      0.0
engagement_rate             inf
scroll_rate                 inf
dtype: float64


###  Number 3 — The gap is large enough to matter


In [38]:
under_perf = top3_volume[top3_volume["perf_group"] == "low"].copy()

current_clicks = under_perf["clicks_90d"].sum()
expected_clicks = under_perf["expected_clicks"].sum()

gap = expected_clicks - current_clicks

total_clicks_all = df["clicks_90d"].sum()
gap_pct_of_total = gap / total_clicks_all * 100

print("=== Gap Analysis ===")
print(f"Underperforming pages: {len(under_perf):,}")
print(f"Current clicks from them: {current_clicks:,.0f}")
print(f"Expected clicks (if typical): {expected_clicks:,.0f}")
print(f"Click gap (missing clicks): {gap:,.0f}")
print(f"Gap as % of all clicks in dataset: {gap_pct_of_total:.1f}%")

print("\n=== Potential Uplift Scenarios ===")
print(f"If we recover 100% of gap: +{gap:,.0f} clicks")
print(f"If we recover 50% of gap: +{gap * 0.5:,.0f} clicks")
print(f"If we recover 25% of gap: +{gap * 0.25:,.0f} clicks")


clicks_per_page_current = under_perf["clicks_90d"].mean()
clicks_per_page_expected = under_perf["expected_clicks"].mean()
clicks_per_page_gap = clicks_per_page_expected - clicks_per_page_current

print(f"\n=== Per-page averages ===")
print(f"Avg clicks per underperf page (current): {clicks_per_page_current:.1f}")
print(f"Avg clicks per underperf page (expected): {clicks_per_page_expected:.1f}")
print(f"Avg gap per page: {clicks_per_page_gap:.1f}")


=== Gap Analysis ===
Underperforming pages: 303
Current clicks from them: 2,457
Expected clicks (if typical): 11,536
Click gap (missing clicks): 9,079
Gap as % of all clicks in dataset: 1.9%

=== Potential Uplift Scenarios ===
If we recover 100% of gap: +9,079 clicks
If we recover 50% of gap: +4,539 clicks
If we recover 25% of gap: +2,270 clicks

=== Per-page averages ===
Avg clicks per underperf page (current): 8.1
Avg clicks per underperf page (expected): 38.1
Avg gap per page: 30.0


### Number 3: The gap is large enough to matter

**Finding:** The 303 underperforming top-3 pages currently generate **2,457 clicks**
but should generate **11,536 clicks** based on typical top-3 CTR.
The **click gap is 9,079 clicks** (1.9% of all dataset clicks).

**Per-page impact:** Each underperforming page averages **8.1 clicks** vs.
an expected **38.1 clicks** — a gap of **30 clicks per page**.

**Potential uplift:**
- Recover 100% of gap: **+9,079 clicks**
- Recover 50% of gap: **+4,539 clicks**  
- Recover 25% of gap: **+2,270 clicks**

**Interpretation:** Even a modest recovery (25–50%) would add thousands of clicks.
The gap is not noise — it is a real, measurable opportunity that justifies
building a model to identify and explain these underperforming pages.

In [39]:
has_ga4 = top3_volume[top3_volume["sessions_90d"] > 0].copy()

comparison_real = has_ga4.groupby("perf_group")[signal_cols].median()
print(comparison_real.T)

perf_group                  high      low    medium
word_count               2884.00   2864.5   2872.00
char_count              19861.00  19707.5  19469.00
content_age_days          329.00    151.0    230.00
days_since_last_update     22.00     22.0     20.00
engagement_rate             2.75      0.0      0.76
scroll_rate                 5.56      0.0      6.00


## 4. Careful Words: What I Can and Can't Claim

### What I CAN Claim

**Observed associations only:**
- Pages with higher CTR in top-3 positions are **associated with** older content
  (median 329 days vs. 151 days). This is an observed pattern, not proof that
  age causes better performance.
- Underperforming pages (bottom 25% CTR) represent 56.8% of top-3 pages with
  meaningful traffic, with a measurable click gap of ~9,079 clicks.
- Underperforming pages show zero median engagement_rate and scroll_rate.
  This may reflect missing GA4 data for some clients, not necessarily zero
  actual engagement.

**Directional guidance:**
- Content age appears more strongly associated with CTR performance than
  word_count or char_count (which show &lt;1% median difference between
  high and low performers).
- The model can help prioritize which pages to review first; the final
  decision still belongs to the content strategist.

**Decision-support:**
- The output ranks candidates for human review. It does not replace human
  judgment about whether a page is truly fixable.

### What I CANNOT Claim

**Causality:**
- I cannot claim that "older content causes better CTR" or that "increasing
  word_count will improve clicks." The data shows word_count differs by only
  0.7% between high and low performers — this is not a meaningful driver.
- I cannot claim that fixing any specific signal will close the gap. I only
  observe which signals travel with high vs. low performance.

**Google's algorithm:**
- I am not reverse-engineering Google's ranking algorithm. I am analyzing
  observable search performance data from FlyRank's own tracking.

**Guaranteed outcomes:**
- I cannot promise a specific traffic increase. The 9,079 click gap is a
  measured observation, not a guaranteed recoverable amount.

**Data completeness:**
- The zero engagement_rate and scroll_rate in the low group may reflect
  missing GA4 data (some clients have no analytics history) rather than
  genuinely zero engagement. I will not claim "underperforming pages have
  no engagement" without a missing-data audit.

### Why This Matters

Public-safety language protects FlyRank, protects clients, and protects the
credibility of the work. Every claim must survive the question:
*"Would I say this in a client meeting with a lawyer in the room?"*

The honest finding so far is surprising: word_count shows almost no
association with CTR gap in this data. The stronger observed signal is
content_age. I will follow the data where it leads, not where my initial
hypothesis expected it to go.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.